In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dune_client.client import DuneClient

dune = DuneClient("qtzqsAdyIntGAWzK68HowaclwlS5BPSD")

query_result = dune.get_latest_result(7337595)
df = pd.DataFrame(query_result.result.rows)
print(df.shape)
df.head()

(5000, 7)


,amount_usd,block_time,contract_address,receiver,sender,symbol,tx_hash
0,1.000423e+09,2026-04-18 09:05:23.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,0x5754284f345afc66a98fbb0a0afe71e0f007b949,0xc6cde7c39eb2f0f0095f41570af89efc2c1ea828,USDT,0xaf6955bc92e72410dc6109352b44de1cefb331163063...
1,6.363376e+08,2026-04-09 01:23:59.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,0xf977814e90da44bfa03b6295a0616a897441acec,0x28c6c06298d514db089934071355e5743bf21d60,USDT,0xb347f190d1e42619bdc4f2ebbc67f30c112f9652dd7e...
2,6.179279e+08,2026-04-14 19:24:23.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,0xf977814e90da44bfa03b6295a0616a897441acec,0x28c6c06298d514db089934071355e5743bf21d60,USDT,0xd77b35877e00c643bd1858d0f1de90143c5105ae7d70...
3,6.000689e+08,2026-03-22 10:07:47.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,0x559432e18b281731c054cd703d4b49872be4ed53,0x6fb624b48d9299674022a23d92515e76ba880113,USDT,0xa14f8d9ab4f24338ed091c90c958a36637b6ddd46386...
4,6.000689e+08,2026-03-22 10:11:47.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,0x559432e18b281731c054cd703d4b49872be4ed53,0xf59869753f41db720127ceb8dbb8afaf89030de4,USDT,0x5db0d231d51905e58e8c53f3036a2ce43f434ea040b3...


In [5]:
df['amount_usd'] = df['amount_usd'].astype(float)
df['block_time'] = pd.to_datetime(df['block_time'])

print("=== Dataset Overview ===")
print(f"Total transactions: {len(df)}")
print(f"Total volume: ${df['amount_usd'].sum()/1e9:.2f}B")
print(f"Avg transfer: ${df['amount_usd'].mean()/1e6:.1f}M")
print(f"Max transfer: ${df['amount_usd'].max()/1e9:.2f}B")
print(f"\nBy stablecoin:")
print(df.groupby('symbol')['amount_usd'].agg(['sum', 'count']).apply(lambda x: x))

=== Dataset Overview ===
Total transactions: 5000
Total volume: $798.51B
Avg transfer: $159.7M
Max transfer: $1.00B

By stablecoin:
                 sum  count
symbol                     
DAI     6.179387e+10    207
PYUSD   2.463056e+09      9
USDC    6.943799e+11   4567
USDT    3.987426e+10    217


In [6]:
threshold = df['amount_usd'].quantile(0.99)
print(f"99th percentile threshold: ${threshold/1e6:.1f}M")

df['segment'] = df['amount_usd'].apply(
    lambda x: 'Mega Transfer' if x >= threshold else 'Large Transfer'
)

print(df['segment'].value_counts())
print(f"\nMega Transfer volume: ${df[df['segment']=='Mega Transfer']['amount_usd'].sum()/1e9:.1f}B")
print(f"Large Transfer volume: ${df[df['segment']=='Large Transfer']['amount_usd'].sum()/1e9:.1f}B")

99th percentile threshold: $406.8M
segment
Large Transfer    4950
Mega Transfer       50
Name: count, dtype: int64

Mega Transfer volume: $23.4B
Large Transfer volume: $775.1B


In [7]:
wallet_volume = df.groupby('sender')['amount_usd'].sum()
total_volume = wallet_volume.sum()
market_shares = wallet_volume / total_volume
hhi = (market_shares ** 2).sum()

print(f"Total unique senders: {len(wallet_volume)}")
print(f"HHI: {hhi:.4f}")

if hhi < 0.15:
    print("Market structure: Competitive")
elif hhi < 0.25:
    print("Market structure: Moderate concentration")
else:
    print("Market structure: High concentration")

top5_share = market_shares.nlargest(5).sum()
print(f"\nTop 5 wallets control: {top5_share*100:.1f}% of volume")

Total unique senders: 120
HHI: 0.1869
Market structure: Moderate concentration

Top 5 wallets control: 72.4% of volume


In [8]:
def gini(values):
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cumsum = np.cumsum(sorted_vals)
    return (2 * np.sum((np.arange(1, n+1) * sorted_vals)) / (n * cumsum[-1])) - (n + 1) / n

gini_coef = gini(wallet_volume.values)
print(f"Gini coefficient: {gini_coef:.4f}")

if gini_coef < 0.4:
    print("Distribution: Relatively equal")
elif gini_coef < 0.6:
    print("Distribution: Moderate inequality")
else:
    print("Distribution: High inequality")

Gini coefficient: 0.8998
Distribution: High inequality


In [9]:
df['z_score'] = (df['amount_usd'] - df['amount_usd'].mean()) / df['amount_usd'].std()

anomalies = df[df['z_score'] > 3].sort_values('z_score', ascending=False)

print(f"Anomalies detected: {len(anomalies)}")
print(f"\nTop 5 anomalous transfers:")
print(anomalies[['block_time', 'symbol', 'amount_usd', 'z_score']].head())

Anomalies detected: 88

Top 5 anomalous transfers:
                 block_time symbol    amount_usd    z_score
0 2026-04-18 09:05:23+00:00   USDT  1.000423e+09  12.600319
1 2026-04-09 01:23:59+00:00   USDT  6.363376e+08   7.143584
2 2026-04-14 19:24:23+00:00   USDT  6.179279e+08   6.867668
3 2026-03-22 10:07:47+00:00   USDT  6.000689e+08   6.600006
4 2026-03-22 10:11:47+00:00   USDT  6.000689e+08   6.600006


In [10]:
from scipy import stats

df_daily = df.set_index('block_time').resample('D')['amount_usd'].sum() / 1e9

autocorr_1 = df_daily.autocorr(lag=1)
autocorr_7 = df_daily.autocorr(lag=7)

print(f"Autocorrelation lag 1 day: {autocorr_1:.4f}")
print(f"Autocorrelation lag 7 days: {autocorr_7:.4f}")

if abs(autocorr_1) > 0.5:
    print("Strong daily pattern detected")
elif abs(autocorr_1) > 0.3:
    print("Moderate daily pattern detected")
else:
    print("No strong daily pattern")
    

Autocorrelation lag 1 day: 0.2346
Autocorrelation lag 7 days: -0.3212
No strong daily pattern


In [11]:
df_daily_stats = df.set_index('block_time').resample('D').agg(
    volume=('amount_usd', 'sum'),
    count=('amount_usd', 'count'),
    avg_size=('amount_usd', 'mean')
) / [1e9, 1, 1e6]

corr, pvalue = stats.pearsonr(df_daily_stats['volume'], df_daily_stats['count'])

print(f"Correlation volume vs count: {corr:.4f}")
print(f"P-value: {pvalue:.4f}")

if pvalue < 0.05:
    print("Statistically significant correlation")
else:
    print("No significant correlation")
    

Correlation volume vs count: 0.9576
P-value: 0.0000
Statistically significant correlation


In [12]:
mega = df[df['segment'] == 'Mega Transfer']['amount_usd']
large = df[df['segment'] == 'Large Transfer']['amount_usd']

t_stat, p_value = stats.ttest_ind(mega, large)

print(f"Mega Transfer — mean: ${mega.mean()/1e6:.0f}M, std: ${mega.std()/1e6:.0f}M")
print(f"Large Transfer — mean: ${large.mean()/1e6:.0f}M, std: ${large.std()/1e6:.0f}M")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Groups are statistically different")
    

Mega Transfer — mean: $469M, std: $101M
Large Transfer — mean: $157M, std: $58M

t-statistic: 37.1886
p-value: 0.000000
Groups are statistically different


In [13]:
df.to_csv('stablecoin_data.csv', index=False)
print('saved!')

saved!
